# Lab: TLS, Certificados E HTTPS

Este notebook guia a execucao do lab local de TLS. A trilha principal usa Docker para subir dois servidores didaticos:

- HTTP em `http://localhost:8080`;
- HTTPS em `https://localhost:8443` com certificado emitido por uma CA local do proprio lab.

O objetivo e entender certificado, chave privada, CA, cadeia de confianca, SAN, handshake TLS e a relacao com Let's Encrypt.

## 1. Preparacao

Execute este notebook a partir da raiz do repositorio `tls-certificados`.

Pre-requisitos locais:

- Python 3;
- Docker;
- OpenSSL;
- curl.

No Colab, use este notebook como roteiro conceitual. A parte com Docker deve ser executada em maquina local ou VM.

In [ ]:
from pathlib import Path

LAB_DIR = Path.cwd()
print('Diretorio atual:', LAB_DIR)
print('Arquivos principais existem?')
for path in ['docker-compose.yml', 'scripts/generate_certs.py', 'app/server.py']:
    print(path, (LAB_DIR / path).exists())

## 2. Gerar CA Local E Certificado Para `localhost`

A CA local representa uma autoridade certificadora didatica. Ela assina o certificado do servidor `localhost`.

Em producao, uma CA publica como Let's Encrypt faria esse papel depois de validar que voce controla o dominio.

In [ ]:
!python3 scripts/generate_certs.py --output-dir certs

## 3. Inspecionar O Certificado

Observe `Subject`, `Issuer`, validade e SAN. O SAN precisa conter `DNS:localhost` e `IP Address:127.0.0.1` para este lab.

In [ ]:
!python3 scripts/inspect_cert.py certs/localhost.crt

In [ ]:
!openssl x509 -in certs/localhost.crt -noout -text | sed -n '/Subject:/p;/Issuer:/p;/Not Before/p;/Not After/p;/Subject Alternative Name/,+1p'

## 4. Subir Os Servidores

Execute a celula abaixo para subir os servicos em segundo plano. Se preferir ver logs ao vivo, rode `docker compose up` em um terminal separado.

In [ ]:
!docker compose up -d

## 5. Comparar HTTP E HTTPS

O primeiro teste usa HTTP simples. Nao ha certificado nem handshake TLS.

In [ ]:
!curl -v http://localhost:8080/

Agora teste HTTPS sem informar a CA local. A conexao deve falhar porque o cliente nao confia automaticamente na CA criada pelo lab.

In [ ]:
!curl -v https://localhost:8443/

Agora informe explicitamente a CA local com `--cacert`. A validacao deve funcionar.

In [ ]:
!curl --cacert certs/lab-ca.crt -v https://localhost:8443/

## 6. Observar O Handshake TLS

`openssl s_client` mostra informacoes do handshake, certificado apresentado e resultado da verificacao.

In [ ]:
!echo | openssl s_client -connect localhost:8443 -servername localhost -CAfile certs/lab-ca.crt 2>/dev/null | sed -n '/subject=/p;/issuer=/p;/Verify return code/p;/Protocol/p;/Cipher/p'

## 7. Relacao Com Let's Encrypt

Let's Encrypt emite certificados publicamente confiaveis depois de validar controle de dominio via ACME. Em um ambiente real, voce normalmente usaria HTTP-01, DNS-01 ou TLS-ALPN-01.

Este lab nao tenta emitir certificado publico para `localhost`, porque `localhost` nao prova controle de um dominio publico. A CA local existe para demonstrar a mecanica sem depender de DNS, porta 80 publica ou dominio real.

## 8. Encerrar O Ambiente

Quando terminar, derrube os containers.

In [ ]:
!docker compose down